<a href="https://colab.research.google.com/github/matthew-ngzc/AI-Safety-Module/blob/main/Week_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Clone the repository

In [ ]:
try:
    ! git clone https://github.com/NayMyatMin/CS427_SMU
    HOME_DIR = "./CS427_SMU/week6/"
except:
    print('Already clone!!!')

fatal: destination path 'CS427_SMU' already exists and is not an empty directory.


# Exercise 4
In this exercise, we aim to simulate how ADF works to find discriminative instances.

In [ ]:
from math import fabs
from operator import truediv
import torch

from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from torch.utils.data import DataLoader

import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import ast

class CensusNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(13, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 8)
        self.fc5 = nn.Linear(8, 4)
        self.fc6 = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        x = F.relu(x)
        x = self.fc4(x)
        x = F.relu(x)
        x = self.fc5(x)
        x = F.relu(x)
        x = self.fc6(x)
        output = x # cross entropy in pytorch already includes softmax
        return output


def load_model(model_class, name):
    model = model_class()
    model.load_state_dict(torch.load(name))

    return model

def isDiscriminative(x, model):
    print('Sample x is: {}'.format(x[0].tolist()))
    x.requires_grad = True
    pred_x = model(x)
    #print('Prediction of x is: {}'.format(pred_x[0].tolist()))
    #print('Label of x is: {}'.format(torch.argmax(pred_x).item()))

    xp = x.detach().clone()
    xp[0][8] = 1-xp[0][8]
    print('Sample xp is: {}'.format(xp[0].tolist()))
    pred_xp = model(xp)
    #print('Prediction of xp is: {}'.format(pred_xp[0].tolist()))
    #print('Label of xp is: {}'.format(torch.argmax(pred_xp).item()))
    if torch.argmax(pred_x).item() != torch.argmax(pred_xp).item():
        print('This sample is discriminary.')
        return True
    else:
        print('This sample is NOT discriminary.')
        return False

device = 'cpu'

model = load_model(CensusNet, HOME_DIR + 'exercise3/census.pt')
labels = np.array(ast.literal_eval(open(HOME_DIR + 'exercise3/census/data/labels.txt', 'r').readline()))
labels = torch.Tensor(labels).type(torch.LongTensor)

correct = 0
for i in range(1):
    file_name = HOME_DIR + 'exercise3/census/data/data' + str(i) + '.txt'
    x = np.array(ast.literal_eval(open(file_name, 'r').readline()))
    x = x.reshape(1, 13)
    x = torch.Tensor(x)

    #The following captures how the global search works, i.e., to identify different seed samples.
    print("Starting global search ...")
    while isDiscriminative(x,model)==False:
        x.requires_grad = True
        pred_x = model(x)
        loss = F.cross_entropy(pred_x, labels)
        loss.backward()
        print('Gradient of x: {}'.format(x.grad.data.sign()[0].tolist()))
        x = x.detach().clone()
        x = x + x.grad.data.sign()

    print("Discriminative instance found.")

    print("\nStarting local search ...")
    #The following examplifies how local search works, i.e., adversarial perturbation.
    print('Seed x is: {}'.format(x[0].tolist()))
    x.requires_grad = True
    pred_x = model(x)
    loss = F.cross_entropy(pred_x, labels)
    loss.backward()
    grad = x.grad.data
    print('Gradient of x: {}'.format(grad.tolist()))
    x = x.detach().clone()
    #TODO: Based on the above-printed grad, update x according to the idea of the local search to find one discriminary instance.
    #Your code goes here. Replace the follwing line with a statement that changes x to produce another instance that is likely discriminary.
    x[0][3] = x[0][3] + 0.1
    #print("You haven't changed x.")
    isDiscriminative(x, model)

Starting global search ...
Sample x is: [5.0, 4.0, 10.0, 0.0, 0.0, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0, 40.0, 0.0]
Sample xp is: [5.0, 4.0, 10.0, 0.0, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 40.0, 0.0]
This sample is discriminary.
Discriminative instance found.

Starting local search ...
Seed x is: [5.0, 4.0, 10.0, 0.0, 0.0, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0, 40.0, 0.0]
Gradient of x: [[0.04739958047866821, -0.256397545337677, 0.3527854382991791, 0.018484199419617653, 0.19417865574359894, -0.9116165637969971, -0.379288911819458, -0.5411887168884277, -0.806613028049469, 1.679221272468567, -0.159250870347023, -0.056537531316280365, -0.5385741591453552]]
Sample x is: [5.0, 4.0, 10.0, 0.10000000149011612, 0.0, 0.0, 2.0, 0.0, 1.0, 0.0, 0.0, 40.0, 0.0]
Sample xp is: [5.0, 4.0, 10.0, 0.10000000149011612, 0.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 40.0, 0.0]
This sample is discriminary.


Look for the lower gradient cos it is the one that will make the smallest change, then change those. Idea is to change only abit so it is still in the vicinity

The 0.018484199419617653 is quite small, so change that one